<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_02_broadcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-02: Broadcasting & Element-wise Ops

**Difficulty**: ⬜ Warm-up

**Objective**: Understand when PyTorch can automatically expand tensors to match shapes, and when it can't.

## Broadcasting Rules
1. Align shapes from the RIGHT
2. Dimensions are compatible if: equal, OR one of them is 1
3. Missing dimensions are treated as 1

```
     (3, 4) + (4,)     → (3, 4)  ✅  (4,) becomes (1, 4) then broadcast
     (3, 4) + (3, 1)   → (3, 4)  ✅  1 broadcasts to 4
     (3, 4) + (3,)     → ERROR   ❌  4 ≠ 3 from right alignment
     (2, 3, 4) + (3, 4) → (2, 3, 4) ✅
```

In [ ]:
import torch

## Task 1: Predict Broadcasting Results

For each pair, predict: will it broadcast? If yes, what's the output shape?

In [ ]:
# TODO: For each pair, first write your prediction as a comment, then verify

a = torch.randn(5, 3)
b = torch.randn(3)
# Prediction: ???
print(f"(5,3) + (3,) = {(a + b).shape}")

c = torch.randn(5, 1)
d = torch.randn(1, 3)
# Prediction: ???
print(f"(5,1) + (1,3) = {(c + d).shape}")

e = torch.randn(2, 3, 4)
f = torch.randn(4)
# Prediction: ???
print(f"(2,3,4) + (4,) = {(e + f).shape}")

g = torch.randn(2, 3, 4)
h = torch.randn(1, 1, 4)
# Prediction: ???
print(f"(2,3,4) + (1,1,4) = {(g + h).shape}")

# This one will FAIL — predict why before uncommenting
# i = torch.randn(3, 4)
# j = torch.randn(3)
# print((i + j).shape)  # Why does this fail?

(5,3) + (3,) = torch.Size([5, 3])
(5,1) + (1,3) = torch.Size([5, 3])
(2,3,4) + (4,) = torch.Size([2, 3, 4])
(2,3,4) + (1,1,4) = torch.Size([2, 3, 4])


## Task 2: Implement Batch Bias Addition

A linear layer computes `y = x @ W.T + b`. The bias `b` has shape `(D_out,)` but `y` has shape `(B, N, D_out)`. Make this work.

In [ ]:
B, N, D_in, D_out = 4, 10, 64, 128
x = torch.randn(B, N, D_in)
W = torch.randn(D_out, D_in)
b = torch.randn(D_out)

# TODO: Compute y = x @ W.T + b using broadcasting (no loops, no reshape of b needed)
y = x @ W.T + b  # YOUR CODE HERE

assert y.shape == (B, N, D_out)
print("✅ Task 2 passed!")

✅ Task 2 passed!


## Task 3: Row-wise and Column-wise Operations

Given a matrix, subtract the row mean and divide by the row std (this is essentially LayerNorm!).

In [ ]:
x = torch.randn(5, 10)

# TODO: Compute row means — shape should be (5, 1) for broadcasting
row_mean = x.mean(dim=1, keepdim=True)  # YOUR CODE

# TODO: Compute row std — shape should be (5, 1)
row_std = x.std(dim=1, keepdim=True)  # YOUR CODE

# TODO: Normalize: (x - mean) / (std + eps)
eps = 1e-5
normalized = (x - row_mean) / (row_std + eps)  # YOUR CODE

assert row_mean.shape == (5, 1)
assert normalized.shape == (5, 10)
# Each row should now have mean ≈ 0 and std ≈ 1
assert torch.allclose(normalized.mean(dim=-1), torch.zeros(5), atol=1e-5)
print("✅ Task 3 passed! You just implemented the core of LayerNorm.")

✅ Task 3 passed! You just implemented the core of LayerNorm.


## Task 4: Outer Product via Broadcasting

Compute the outer product of two vectors using broadcasting (no `torch.outer`).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])  # shape (3,)
b = torch.tensor([4.0, 5.0])        # shape (2,)

# TODO: Compute outer product using broadcasting
# Hint: reshape a to (3, 1) and b to (1, 2), then multiply
outer = a.reshape(3, 1) @ b.reshape(1, 2)  # YOUR CODE — should be shape (3, 2)

expected = torch.tensor([[4., 5.], [8., 10.], [12., 15.]])
assert torch.allclose(outer, expected)
print("✅ Task 4 passed!")

✅ Task 4 passed!


## Task 5: Pairwise Distance Matrix

Given N points in D dimensions, compute the NxN distance matrix. This pattern appears in attention scores.

In [41]:
N, D = 5, 3
points = torch.randn(N, D)

# TODO: Compute pairwise L2 distances using broadcasting
# Hint: reshape points to (N, 1, D) and (1, N, D), subtract, then norm
diffs = points.view(N, 1, D) - points.view(1, N, D)
distances = torch.norm(diffs, p=2, dim=-1)  # YOUR CODE — shape (N, N)

assert distances.shape == (N, N)
# Diagonal should be zero (distance to self)
assert torch.allclose(distances.diagonal(), torch.zeros(N), atol=1e-6)
# Should be symmetric
assert torch.allclose(distances, distances.T, atol=1e-6)
print("✅ Task 5 passed!")

✅ Task 5 passed!
